In [1]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load data
with open("work_fields.json", "r", encoding="utf-8") as f:
    data = json.load(f)

ids = [d["correlationMatrixId"] for d in data]
texts = [f"{d['nameDe']} (DE), {d['nameEn']} (EN)" for d in data]
n = len(ids)

# 2. Load model
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

# 3. Encode
embeddings = model.encode(texts, normalize_embeddings=True)

# 4. Similarity
sim_matrix = cosine_similarity(embeddings).astype(np.float64)  # <- ensure float64

top_k = 10
final_result = []

for i in range(n):
    sims = sim_matrix[i]
    top_indices = np.argsort(sims)[::-1][:top_k]

    for rank_position, j in enumerate(top_indices):
        value_rank = int(top_k - rank_position)           # 10..1
        score = float(sims[j])                            # python float

        final_result.append({
            "code1": str(ids[i]),
            "code2": str(ids[j]),
            "value": value_rank,
            "score": round(score, 6)
        })

# 5. Save
with open("correlation_matrix.json", "w", encoding="utf-8") as f:
    json.dump(final_result, f, indent=2, ensure_ascii=False)

print("Done.")
print(f"Total rows: {len(final_result)} (expected {n * top_k})")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done.
Total rows: 1800 (expected 1800)
